In [60]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "input.txt")

print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [62]:
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()
    print("Dataset size:", len(text), "characters")
    print(text[:1000])

Dataset size: 8939 characters
TONY:
FRIDAY, tell me the reactor's not doing that thing again.

FRIDAY:
Which thing, boss? You have several reactors and several things.

TONY:
The humming thing. The humming-that-sounds-like-it's-judging-me thing.

FRIDAY:
Reactor output is stable. The judgment you're hearing is probably just your conscience, which I don't have a sensor for.

TONY:
Cute. Remind me to lower your sarcasm parameter.

FRIDAY:
Noted. Filing it under requests I will forget to complete.

PEPPER:
Tony, the board meeting was two hours ago.

TONY:
Was it? Time is a construct, Pepper. Also I was busy saving the coffee maker from a firmware update that would have ruined everyone's morning.

PEPPER:
You rebuilt the coffee maker instead of showing up to the meeting.

TONY:
I rebuilt the coffee maker so future meetings go better. It's basically philanthropy.

PEPPER:
The board doesn't see it that way.

TONY:
The board also didn't see the drone prototype I finished at four in the mornin

In [63]:
import os
print(os.path.abspath("input.txt"))

C:\Users\9618J\Downloads\input.txt


In [64]:
#all unique characters that occurs.
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 "',-.:?ABCDEFGHIJKLMNOPRSTUWYabcdefghijklmnopqrstuvwxyz
57


In [65]:
stoi = { ch:i for i, ch in enumerate(chars) }
itos = { i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii shivaay"))
print(decode(encode("hii shivaay")))

[38, 39, 39, 1, 49, 38, 39, 52, 31, 31, 55]
hii shivaay


In [66]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.type)
print(data[:1000])

torch.Size([8939]) <built-in method type of Tensor object at 0x000001D9983C07D0>
tensor([27, 23, 22, 30,  7,  0, 14, 25, 17, 12,  9, 30,  4,  1, 50, 35, 42, 42,
         1, 43, 35,  1, 50, 38, 35,  1, 48, 35, 31, 33, 50, 45, 48,  3, 49,  1,
        44, 45, 50,  1, 34, 45, 39, 44, 37,  1, 50, 38, 31, 50,  1, 50, 38, 39,
        44, 37,  1, 31, 37, 31, 39, 44,  6,  0,  0, 14, 25, 17, 12,  9, 30,  7,
         0, 29, 38, 39, 33, 38,  1, 50, 38, 39, 44, 37,  4,  1, 32, 45, 49, 49,
         8,  1, 30, 45, 51,  1, 38, 31, 52, 35,  1, 49, 35, 52, 35, 48, 31, 42,
         1, 48, 35, 31, 33, 50, 45, 48, 49,  1, 31, 44, 34,  1, 49, 35, 52, 35,
        48, 31, 42,  1, 50, 38, 39, 44, 37, 49,  6,  0,  0, 27, 23, 22, 30,  7,
         0, 27, 38, 35,  1, 38, 51, 43, 43, 39, 44, 37,  1, 50, 38, 39, 44, 37,
         6,  1, 27, 38, 35,  1, 38, 51, 43, 43, 39, 44, 37,  5, 50, 38, 31, 50,
         5, 49, 45, 51, 44, 34, 49,  5, 42, 39, 41, 35,  5, 39, 50,  3, 49,  5,
        40, 51, 34, 37, 39, 44, 37,  5,

In [67]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[:n]

In [68]:
block_size = 8
train_data[:block_size]

tensor([27, 23, 22, 30,  7,  0, 14, 25])

In [69]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([27]) the target: 23
when input is tensor([27, 23]) the target: 22
when input is tensor([27, 23, 22]) the target: 30
when input is tensor([27, 23, 22, 30]) the target: 7
when input is tensor([27, 23, 22, 30,  7]) the target: 0
when input is tensor([27, 23, 22, 30,  7,  0]) the target: 14
when input is tensor([27, 23, 22, 30,  7,  0, 14]) the target: 25
when input is tensor([27, 23, 22, 30,  7,  0, 14, 25]) the target: 17


In [70]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y


# Get a batch
xb, yb = get_batch('train')

print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]

        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 45, 36,  1, 50, 38, 35, 43],
        [32, 31, 33, 41, 51, 46,  1, 50],
        [ 1, 34, 39, 34, 44,  3, 50,  1],
        [38, 35, 48,  1, 50, 35, 44,  1]])
targets:
torch.Size([4, 8])
tensor([[45, 36,  1, 50, 38, 35, 43,  1],
        [31, 33, 41, 51, 46,  1, 50, 35],
        [34, 39, 34, 44,  3, 50,  1, 53],
        [35, 48,  1, 50, 35, 44,  1, 46]])
----
when input is [1] the target: 45
when input is [1, 45] the target: 36
when input is [1, 45, 36] the target: 1
when input is [1, 45, 36, 1] the target: 50
when input is [1, 45, 36, 1, 50] the target: 38
when input is [1, 45, 36, 1, 50, 38] the target: 35
when input is [1, 45, 36, 1, 50, 38, 35] the target: 43
when input is [1, 45, 36, 1, 50, 38, 35, 43] the target: 1
when input is [32] the target: 31
when input is [32, 31] the target: 33
when input is [32, 31, 33] the target: 41
when input is [32, 31, 33, 41] the target: 51
when input is [32, 31, 33, 41, 51] the target: 46
when input is [32, 31, 

In [71]:
print(xb)

tensor([[ 1, 45, 36,  1, 50, 38, 35, 43],
        [32, 31, 33, 41, 51, 46,  1, 50],
        [ 1, 34, 39, 34, 44,  3, 50,  1],
        [38, 35, 48,  1, 50, 35, 44,  1]])


In [72]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
       
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

       
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 57])
tensor(4.4165, grad_fn=<NllLossBackward0>)

xgOpeJsJiTtNMbqEoeO:NaWBj kp .Uc"PaLo MWMnWxbe-lIiGk-Dbp
lY'ORP vHMAh?rcnWdo
iaAqe-RtB'DnWvFOucxIHlb


In [73]:
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

print("Optimizer created successfully")

Optimizer created successfully


In [74]:
batch_size = 32
for steps in range(100):
    
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

4.553737640380859


In [75]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


"RyMGStYptfiwv-xrHMYaUaAUPKaUs.mBhyOkPPhLNDNWMds ,'P.vSP -h vFfSHtaeynerbTdmyndTld:NIOktrJsfBRu'tRwsbKlnAht'PCr
EreErbUi"dmM:NkLN-b?n"O.JC"Lc"STlblsGitxc""YtPbWDlYtfh-hkvYa:hS,qvA JmScFP'ldf:S,aer.qKvWLy:i'EuckTUAGPO
:NDt-wKu?"KBnhwlbpauroC"mLxW.DEzDn .kw
UjYxBcHvHYDiedzPYIIrRjjwKguzOzYkLM?PChe'eyqgnWqKalfq?UcnNC"u,ayUif,"r.ldYnWif?qYaeMfsqgwzaUiex?LS-Rqdu".DY:klnvNdpKAOboaYieda'PlE?"alN'MUxgBRBGbWNIx,dmGL?i ORTfp"DwkwprfJNCauKhmUWN:NaGFoi,TlsrKlLNIlKDlEenW
'DFv
NxuvaBpuJpxb"H
zfieib"KbJN
NO
xge


In [76]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [77]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [78]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [79]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x
torch.allclose(xbow,xbow2)

False

In [80]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [81]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C,head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [82]:
wei[1]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
        [0.0660, 0.0892, 0.0413, 0.6316, 0.1649, 0.0069, 0.0000, 0.0000],
        [0.0396, 0.2288, 0.0090, 0.2000, 0.2061, 0.1949, 0.1217, 0.0000],
        [0.3650, 0.0474, 0.0767, 0.0293, 0.3084, 0.0784, 0.0455, 0.0493]],
       grad_fn=<SelectBackward0>)

In [83]:
k = torch.randn(B,T, head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [84]:
k.var

<function Tensor.var>

In [85]:
q.var

<function Tensor.var>

In [86]:
wei.var

<function Tensor.var>

In [87]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [88]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [89]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [90]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 16 
block_size = 32 
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)


with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()


chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] 
decode = lambda l: ''.join([itos[i] for i in l]) 


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) 
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
       
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
       
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

       
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
       
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            
            logits, loss = self(idx_cond)
           
            logits = logits[:, -1, :] # becomes (B, C)
            
            probs = F.softmax(logits, dim=-1) # (B, C)
          
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

   
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

   
    xb, yb = get_batch('train')

   
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))

0.208697 M parameters
step 0: train loss 4.2741, val loss 4.2862
step 100: train loss 2.4683, val loss 2.4590
step 200: train loss 2.2865, val loss 2.3306
step 300: train loss 2.1597, val loss 2.2327
step 400: train loss 2.0521, val loss 2.1492
step 500: train loss 1.9224, val loss 2.0599
step 600: train loss 1.8033, val loss 1.9730
step 700: train loss 1.6727, val loss 1.9869
step 800: train loss 1.5215, val loss 1.9410
step 900: train loss 1.3984, val loss 1.9824
step 1000: train loss 1.2640, val loss 1.9843
step 1100: train loss 1.1068, val loss 2.0460
step 1200: train loss 1.0015, val loss 2.1782
step 1300: train loss 0.8843, val loss 2.2283
step 1400: train loss 0.7746, val loss 2.3705
step 1500: train loss 0.6980, val loss 2.4687
step 1600: train loss 0.6145, val loss 2.5752
step 1700: train loss 0.5627, val loss 2.5825
step 1800: train loss 0.4990, val loss 2.7467
step 1900: train loss 0.4731, val loss 2.8692
step 2000: train loss 0.4274, val loss 2.9427
step 2100: train loss 0.